# 🎬 Talking Avatar — Türkçe Animasyon/Film Üretme (Colab)

Karakter görseli + metin → **Türkçe seslendirme (Coqui XTTS v2)** + **dudak senkronu (Wav2Lip)** + **FFmpeg montaj** → tek `mp4`.

**Kullanım:** Hücreleri **sırayla** çalıştır. Başlamadan: `Runtime → Change runtime type → T4 GPU`.

> ⚠️ En kırılgan adım **Hücre 2** (Wav2Lip model indirme). Linkler değişebilirse README'deki nota bak.

In [ ]:
# HÜCRE 0 — Ortam ve klasörler
import os, subprocess
gpu = subprocess.getoutput("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
print("GPU:", gpu if gpu and "failed" not in gpu.lower() else "⚠️ GPU YOK! Runtime > Change runtime type > T4 GPU")
for d in ["assets", "audio", "clips", "output"]:
    os.makedirs(d, exist_ok=True)
print("Klasörler hazır.")

In [ ]:
# HÜCRE 1 — Seslendirme motoru (XTTS v2, Türkçe, çok sesli)
import os
os.environ["COQUI_TOS_AGREED"] = "1"
!pip -q install coqui-tts

from TTS.api import TTS
import torch
_device = "cuda" if torch.cuda.is_available() else "cpu"
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(_device)
print("✅ XTTS yüklendi. Cihaz:", _device)

try:
    isimler = list(tts.synthesizer.tts_model.speaker_manager.name_to_id.keys())
    print("Toplam hazır ses:", len(isimler), "| örnek:", isimler[:8])
except Exception as e:
    print("Ses listesi okunamadı (sorun değil):", e)

In [ ]:
# HÜCRE 2 — Konuşturma motoru (Wav2Lip)  ⚠️ kırılgan adım
import os
if not os.path.exists("Wav2Lip"):
    !git clone -q https://github.com/justinjohn0306/Wav2Lip
%cd Wav2Lip
!mkdir -p checkpoints
!wget -q "https://github.com/justinjohn0306/Wav2Lip/releases/download/models/wav2lip_gan.pth" -O checkpoints/wav2lip_gan.pth
!wget -q "https://github.com/justinjohn0306/Wav2Lip/releases/download/models/mobilenet.pth" -O checkpoints/mobilenet.pth
!pip -q install librosa==0.10.2 numba batch-face
%cd ..
ok = os.path.exists("Wav2Lip/checkpoints/wav2lip_gan.pth") and os.path.getsize("Wav2Lip/checkpoints/wav2lip_gan.pth") > 1_000_000
print("✅ Wav2Lip hazır" if ok else "❌ Model inmedi — README'deki güncel checkpoint linkini kullan")

In [ ]:
# HÜCRE 3 — Karakter yüz görsellerini yükle (önden bakan, net, tek yüz, 512px+)
from google.colab import files
import shutil, os
up = files.upload()
for fn in list(up.keys()):
    shutil.move(fn, f"assets/{fn}")
print("Yüklenenler:", sorted(os.listdir("assets")))

In [ ]:
# HÜCRE 4 — Karakter + replik formu (ipywidgets)
import ipywidgets as w
from IPython.display import display, clear_output
import json, os

XTTS_SESLER = ["Sofia Hellen","Daisy Studious","Claribel Dervla","Alison Dietlinde",
               "Tanja Adelina","Damien Black","Andrew Chipper","Viktor Eka",
               "Craig Gutsy","Eugenio Mataracı","Ferran Simen","Baldur Sanjin"]

gorseller = sorted(os.listdir("assets"))
characters, lines = {}, []

k_isim=w.Text(description="İsim:"); k_gorsel=w.Dropdown(options=gorseller,description="Görsel:")
k_ses=w.Dropdown(options=XTTS_SESLER,description="Ses:"); k_btn=w.Button(description="Karakter Ekle",button_style="success")
r_char=w.Dropdown(options=[],description="Karakter:")
r_text=w.Textarea(description="Replik:",layout=w.Layout(width="60%",height="60px"))
r_btn=w.Button(description="Replik Ekle",button_style="info")
ornek_btn=w.Button(description="Örnek doldur",button_style="warning")
kaydet_btn=w.Button(description="KAYDET → project.json",button_style="danger")
cikti=w.Output()

def yenile():
    r_char.options=list(characters.keys())
    with cikti:
        clear_output()
        print("KARAKTERLER:");  [print(f"  • {n} [{v['image']} / {v['speaker']}]") for n,v in characters.items()]
        print("\nREPLİKLER:");   [print(f"  {i}. {l['char']}: {l['text']}") for i,l in enumerate(lines,1)]

def karakter_ekle(_):
    if k_isim.value.strip() and k_gorsel.value:
        characters[k_isim.value.strip()]={"image":k_gorsel.value,"speaker":k_ses.value}; k_isim.value=""; yenile()
def replik_ekle(_):
    if r_char.value and r_text.value.strip():
        lines.append({"char":r_char.value,"text":r_text.value.strip()}); r_text.value=""; yenile()
def ornek(_):
    if len(gorseller)<2:
        with cikti: clear_output(); print("Önce en az 2 görsel yükle (Hücre 3).")
        return
    characters.clear(); lines.clear()
    characters["Ayşe"]={"image":gorseller[0],"speaker":"Sofia Hellen"}
    characters["Mert"]={"image":gorseller[1],"speaker":"Damien Black"}
    for c,t in [("Ayşe","Merhaba Mert, bugün hava çok güzel değil mi?"),
                ("Mert","Evet Ayşe, böyle bir günde dışarı çıkmamak olmaz."),
                ("Ayşe","O zaman sahilde yürüyüşe ne dersin?"),
                ("Mert","Harika fikir, hemen hazırlanıyorum!")]:
        lines.append({"char":c,"text":t})
    yenile()
def kaydet(_):
    json.dump({"characters":characters,"lines":lines},open("project.json","w",encoding="utf-8"),ensure_ascii=False,indent=2)
    with cikti: clear_output(); print("✅ Kaydedildi → project.json")

for b,f in [(k_btn,karakter_ekle),(r_btn,replik_ekle),(ornek_btn,ornek),(kaydet_btn,kaydet)]: b.on_click(f)
display(w.HTML("<b>1) Karakter tanımla</b>"), w.HBox([k_isim,k_gorsel,k_ses,k_btn]),
        w.HTML("<b>2) Replik ekle (sırayla)</b>"), w.HBox([r_char,r_text,r_btn]),
        w.HBox([ornek_btn,kaydet_btn]), cikti)

In [ ]:
# HÜCRE 5 — Repliklerden Türkçe ses üret (XTTS v2)
import json
proj=json.load(open("project.json",encoding="utf-8")); chars,lines=proj["characters"],proj["lines"]
for i,ln in enumerate(lines):
    spk=chars[ln["char"]]["speaker"]; out=f"audio/line_{i:03d}.wav"
    tts.tts_to_file(text=ln["text"], speaker=spk, language="tr", file_path=out)
    print(f"[{i:03d}] {ln['char']} ({spk}) → {out}")
print("✅ Tüm sesler hazır.")

In [ ]:
# HÜCRE 6 — Her replik için konuşan klip (Wav2Lip)
import json, os, subprocess
proj=json.load(open("project.json",encoding="utf-8")); chars,lines=proj["characters"],proj["lines"]
for i,ln in enumerate(lines):
    img=f"assets/{chars[ln['char']]['image']}"; wav=f"audio/line_{i:03d}.wav"; out=f"clips/line_{i:03d}.mp4"
    print(f"→ Klip {i:03d}: {ln['char']}")
    cmd=["python","inference.py","--checkpoint_path","checkpoints/wav2lip_gan.pth",
         "--face",f"../{img}","--audio",f"../{wav}","--outfile",f"../{out}",
         "--nosmooth","--pads","0","10","0","0"]
    r=subprocess.run(cmd,cwd="Wav2Lip",capture_output=True,text=True)
    if not os.path.exists(out):
        print(r.stdout[-1500:]); print(r.stderr[-1500:]); raise RuntimeError("Klip üretilemedi (genelde yüz bulunamadı)")
print("✅ Klipler hazır.")

In [ ]:
# HÜCRE 7 — Klipleri birleştir → tek mp4
import os, subprocess
W,H,FPS=720,720,25
clips=sorted(f for f in os.listdir("clips") if f.endswith(".mp4") and not f.startswith("n_"))
norm=[]
for c in clips:
    vf=f"scale={W}:{H}:force_original_aspect_ratio=decrease,pad={W}:{H}:(ow-iw)/2:(oh-ih)/2,setsar=1,fps={FPS}"
    subprocess.run(["ffmpeg","-y","-i",f"clips/{c}","-vf",vf,"-c:v","libx264","-c:a","aac","-ar","24000",f"clips/n_{c}"],capture_output=True)
    norm.append(f"n_{c}")
open("clips/list.txt","w").write("".join(f"file '{n}'\n" for n in norm))
subprocess.run(["ffmpeg","-y","-f","concat","-safe","0","-i","clips/list.txt","-c","copy","output/final.mp4"],capture_output=True)
print("✅ output/final.mp4 —", os.path.getsize("output/final.mp4"), "byte")
from google.colab import files; files.download("output/final.mp4")

In [ ]:
# HÜCRE 7b — (opsiyonel) Altyazı göm
import subprocess, json, os
lines=json.load(open("project.json",encoding="utf-8"))["lines"]
def dur(p): return float(subprocess.run(["ffprobe","-v","error","-show_entries","format=duration","-of","csv=p=0",p],capture_output=True,text=True).stdout)
def ts(s): h=int(s//3600);m=int(s%3600//60);return f"{h:02d}:{m:02d}:{s%60:06.3f}".replace('.',',')
t=0.0; parts=[]; clips=sorted(f for f in os.listdir("clips") if f.startswith("n_"))
for i,(c,ln) in enumerate(zip(clips,lines),1):
    d=dur(f"clips/{c}"); parts.append(f"{i}\n{ts(t)} --> {ts(t+d)}\n{ln['char']}: {ln['text']}\n"); t+=d
open("output/altyazi.srt","w",encoding="utf-8").write("\n".join(parts))
subprocess.run(["ffmpeg","-y","-i","output/final.mp4","-vf","subtitles=output/altyazi.srt:force_style='FontSize=18'","-c:a","copy","output/final_altyazili.mp4"],capture_output=True)
print("✅ output/final_altyazili.mp4")